In [1]:
from pyspark.sql import SparkSession

spark=(SparkSession.builder.appName("Basic_spark").master("local[*]").getOrCreate())

In [2]:
spark

In [112]:
emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29"," ","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

In [7]:
emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

In [113]:
emp=spark.createDataFrame(data=emp_data,schema=emp_schema)

In [9]:
emp.show()

+-----------+-------------+-------------+---+------+------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|
+-----------+-------------+-------------+---+------+------+----------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-08-01|
|        011|          104|   David Park| 38|  Male| 65000|2015-11-01|
|     

In [30]:
emp_final_1=emp.where("salary>50000")

In [33]:
emp_final.write.format("csv").save("data/output/1/emp.csv")

In [35]:
emp_final_1.rdd.getNumPartitions()

4

In [37]:
emp.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: string (nullable = true)



In [44]:
from pyspark.sql.functions import expr,col

emp_2=emp.select(col('salary').cast('int'))


In [45]:
emp_2.printSchema()

root
 |-- salary: integer (nullable = true)



In [48]:
from pyspark.sql.types import *

schema_1=StructType([
         StructField("name",StringType(),True),
         StructField("age",IntegerType(),True)
])

In [50]:
emp_3=emp.selectExpr("employee_id as emp_id","name","cast(age as int) as age","salary")

In [51]:
emp_3.printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: string (nullable = true)



In [53]:
emp_4=emp_3.select("emp_id","name","age","salary").where("age>30")

In [54]:
emp_4.show()

+------+-------------+---+------+
|emp_id|         name|age|salary|
+------+-------------+---+------+
|   003|    Bob Brown| 35| 55000|
|   005|    Jack Chan| 40| 60000|
|   006|    Jill Wong| 32| 52000|
|   007|James Johnson| 42| 70000|
|   009|      Tom Tan| 33| 58000|
|   011|   David Park| 38| 65000|
|   012|   Susan Chen| 31| 54000|
|   013|    Brian Kim| 45| 75000|
|   015|  Michael Lee| 37| 63000|
|   017|  George Wang| 34| 57000|
|   019|  Steven Chen| 36| 62000|
|   020|    Grace Kim| 32| 53000|
+------+-------------+---+------+



In [55]:
emp_4.write.format("csv").save("data/output/2/emp.csv")

In [61]:
from pyspark.sql.types import _parse_datatype_string

schema_str="emp_id string,age int"

schema_spark=_parse_datatype_string(schema_str)


In [62]:
schema_spark

StructType([StructField('emp_id', StringType(), True), StructField('age', IntegerType(), True)])

In [63]:
# Casting Column
# select employee_id, name, age, cast(salary as double) as salary from emp
emp_casted=emp.select("employee_id","name","age",col("salary").cast("double"))

In [64]:
emp_casted.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- salary: double (nullable = true)



In [65]:
emp_taxed=emp_casted.withColumn("tax",col("salary")*0.2)

In [68]:
# Literals
# select employee_id, name, age, salary, tax, 1 as columnOne, 'two' as columnTwo from emp_taxed
from pyspark.sql.functions import lit
emp_new_columns=emp_taxed.withColumn("columnOne",lit(1)).withColumn("columntwo",lit("two"))

In [69]:
emp_new_columns.show()

+-----------+-------------+---+-------+-------+---------+---------+
|employee_id|         name|age| salary|    tax|columnOne|columntwo|
+-----------+-------------+---+-------+-------+---------+---------+
|        001|     John Doe| 30|50000.0|10000.0|        1|      two|
|        002|   Jane Smith| 25|45000.0| 9000.0|        1|      two|
|        003|    Bob Brown| 35|55000.0|11000.0|        1|      two|
|        004|    Alice Lee| 28|48000.0| 9600.0|        1|      two|
|        005|    Jack Chan| 40|60000.0|12000.0|        1|      two|
|        006|    Jill Wong| 32|52000.0|10400.0|        1|      two|
|        007|James Johnson| 42|70000.0|14000.0|        1|      two|
|        008|     Kate Kim| 29|51000.0|10200.0|        1|      two|
|        009|      Tom Tan| 33|58000.0|11600.0|        1|      two|
|        010|     Lisa Lee| 27|47000.0| 9400.0|        1|      two|
|        011|   David Park| 38|65000.0|13000.0|        1|      two|
|        012|   Susan Chen| 31|54000.0|10800.0| 

In [72]:
emp_1=emp_new_columns.withColumnRenamed("employee_id","emp_id")

In [73]:
emp_1.show()

+------+-------------+---+-------+-------+---------+---------+
|emp_id|         name|age| salary|    tax|columnOne|columntwo|
+------+-------------+---+-------+-------+---------+---------+
|   001|     John Doe| 30|50000.0|10000.0|        1|      two|
|   002|   Jane Smith| 25|45000.0| 9000.0|        1|      two|
|   003|    Bob Brown| 35|55000.0|11000.0|        1|      two|
|   004|    Alice Lee| 28|48000.0| 9600.0|        1|      two|
|   005|    Jack Chan| 40|60000.0|12000.0|        1|      two|
|   006|    Jill Wong| 32|52000.0|10400.0|        1|      two|
|   007|James Johnson| 42|70000.0|14000.0|        1|      two|
|   008|     Kate Kim| 29|51000.0|10200.0|        1|      two|
|   009|      Tom Tan| 33|58000.0|11600.0|        1|      two|
|   010|     Lisa Lee| 27|47000.0| 9400.0|        1|      two|
|   011|   David Park| 38|65000.0|13000.0|        1|      two|
|   012|   Susan Chen| 31|54000.0|10800.0|        1|      two|
|   013|    Brian Kim| 45|75000.0|15000.0|        1|   

In [74]:
# column name with space

emp_2=emp_1.withColumnRenamed("columnTwo","column Two")

In [75]:
emp_2.printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- tax: double (nullable = true)
 |-- columnOne: integer (nullable = false)
 |-- column Two: string (nullable = false)



In [76]:
emp_dropped=emp_new_columns.drop("columnTwo","columnOne")


In [77]:
emp_dropped.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- tax: double (nullable = true)



In [78]:
emp_filtered=emp_dropped.where("tax>10000")

In [79]:
emp_filtered.show()

+-----------+-------------+---+-------+-------+
|employee_id|         name|age| salary|    tax|
+-----------+-------------+---+-------+-------+
|        003|    Bob Brown| 35|55000.0|11000.0|
|        005|    Jack Chan| 40|60000.0|12000.0|
|        006|    Jill Wong| 32|52000.0|10400.0|
|        007|James Johnson| 42|70000.0|14000.0|
|        008|     Kate Kim| 29|51000.0|10200.0|
|        009|      Tom Tan| 33|58000.0|11600.0|
|        011|   David Park| 38|65000.0|13000.0|
|        012|   Susan Chen| 31|54000.0|10800.0|
|        013|    Brian Kim| 45|75000.0|15000.0|
|        015|  Michael Lee| 37|63000.0|12600.0|
|        017|  George Wang| 34|57000.0|11400.0|
|        019|  Steven Chen| 36|62000.0|12400.0|
|        020|    Grace Kim| 32|53000.0|10600.0|
+-----------+-------------+---+-------+-------+



In [80]:
emp_limit=emp_filtered.limit(5)

In [81]:
emp_limit.show()

+-----------+-------------+---+-------+-------+
|employee_id|         name|age| salary|    tax|
+-----------+-------------+---+-------+-------+
|        003|    Bob Brown| 35|55000.0|11000.0|
|        005|    Jack Chan| 40|60000.0|12000.0|
|        006|    Jill Wong| 32|52000.0|10400.0|
|        007|James Johnson| 42|70000.0|14000.0|
|        008|     Kate Kim| 29|51000.0|10200.0|
+-----------+-------------+---+-------+-------+



In [83]:
emp_limit.show(2)

+-----------+---------+---+-------+-------+
|employee_id|     name|age| salary|    tax|
+-----------+---------+---+-------+-------+
|        003|Bob Brown| 35|55000.0|11000.0|
|        005|Jack Chan| 40|60000.0|12000.0|
+-----------+---------+---+-------+-------+
only showing top 2 rows



In [85]:
columns={
    "Tax":col("salary")*0.2,
    "ColumnOne":lit(1),
    "columnTwo":lit("two")
    
}

emp_final=emp.withColumns(columns)

In [86]:
emp_final.show()

+-----------+-------------+-------------+---+------+------+----------+-------+---------+---------+
|employee_id|department_id|         name|age|gender|salary| hire_date|    Tax|ColumnOne|columnTwo|
+-----------+-------------+-------------+---+------+------+----------+-------+---------+---------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|10000.0|        1|      two|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15| 9000.0|        1|      two|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|11000.0|        1|      two|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30| 9600.0|        1|      two|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|12000.0|        1|      two|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|10400.0|        1|      two|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|14000.0|        1|      two|
|        0

In [114]:
from pyspark.sql.functions import when
emp_gender_fixed=emp.withColumn("New_gender",when(col("gender")=="Male","M").when(col("gender")=="Female","F").otherwise(None))

In [95]:
emp_gender_fixed_1=emp.withColumn("new_column",expr("case when gender ='Male' then 'M' when gender='Female' then 'F' else null end"))

In [115]:
emp_gender_fixed_1.show()

+-----------+-------------+-------------+---+------+------+----------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|new_column|
+-----------+-------------+-------------+---+------+------+----------+----------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|         M|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         M|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|         F|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|         M|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|         F|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|         M|
|        010|   

In [98]:
from pyspark.sql.functions import regexp_replace

emp_name_fixed=emp_gender_fixed.withColumn("New_name",regexp_replace(col('Name'),'j','z'))

In [100]:
from pyspark.sql.functions import to_date
emp_date_fix=emp_name_fixed.withColumn("hire_date",to_date(col("hire_date"),'yyyy-MM-dd'))

In [102]:
emp_date_fix.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- New_gender: string (nullable = true)
 |-- New_name: string (nullable = true)



In [104]:
from pyspark.sql.functions import current_date,current_timestamp
emp_dated=emp_date_fix.withColumn("current_date",current_date()).withColumn("Time_stamp",current_timestamp())

In [106]:
emp_dated.show(truncate=False)

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------------+
|employee_id|department_id|name         |age|gender|salary|hire_date |New_gender|New_name     |current_date|Time_stamp                |
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------------+
|001        |101          |John Doe     |30 |Male  |50000 |2015-01-01|M         |John Doe     |2026-03-19  |2026-03-19 14:17:35.099107|
|002        |101          |Jane Smith   |25 |Female|45000 |2016-02-15|F         |Jane Smith   |2026-03-19  |2026-03-19 14:17:35.099107|
|003        |102          |Bob Brown    |35 |Male  |55000 |2014-05-01|M         |Bob Brown    |2026-03-19  |2026-03-19 14:17:35.099107|
|004        |102          |Alice Lee    |28 |Female|48000 |2017-09-30|F         |Alice Lee    |2026-03-19  |2026-03-19 14:17:35.099107|
|005        |103          |Jack Chan    |40 |Mal

In [110]:
emp_1=emp_dated.na.drop()

In [111]:
emp_1.show()

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|New_gender|     New_name|current_date|          Time_stamp|
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|     John Doe|  2026-03-19|2026-03-19 14:19:...|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|   Jane Smith|  2026-03-19|2026-03-19 14:19:...|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|         M|    Bob Brown|  2026-03-19|2026-03-19 14:19:...|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|    Alice Lee|  2026-03-19|2026-03-19 14:19:...|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         M|    Jack 

In [117]:
from pyspark.sql.functions import coalesce,lit

emp_null_df=emp_dated.withColumn("New_gender",coalesce(col("new_gender"),lit("o")))

In [119]:
emp_final=emp_null_df.drop("name","gender").withColumnRenamed("New_name","Name").withColumnRenamed("New_gender","Gender")

In [120]:
emp_final.show()

+-----------+-------------+---+------+----------+------+-------------+------------+--------------------+
|employee_id|department_id|age|salary| hire_date|Gender|         Name|current_date|          Time_stamp|
+-----------+-------------+---+------+----------+------+-------------+------------+--------------------+
|        001|          101| 30| 50000|2015-01-01|     M|     John Doe|  2026-03-19|2026-03-19 14:30:...|
|        002|          101| 25| 45000|2016-02-15|     F|   Jane Smith|  2026-03-19|2026-03-19 14:30:...|
|        003|          102| 35| 55000|2014-05-01|     M|    Bob Brown|  2026-03-19|2026-03-19 14:30:...|
|        004|          102| 28| 48000|2017-09-30|     F|    Alice Lee|  2026-03-19|2026-03-19 14:30:...|
|        005|          103| 40| 60000|2013-04-01|     M|    Jack Chan|  2026-03-19|2026-03-19 14:30:...|
|        006|          103| 32| 52000|2018-07-01|     F|    Jill Wong|  2026-03-19|2026-03-19 14:30:...|
|        007|          101| 42| 70000|2012-03-15|     M

In [123]:
df_reordered=emp_final.select(expr("employee_id as emp_id"),"department_id","Name","age","salary","hire_date","gender","Current_date","Time_stamp")

In [124]:
df_reordered.show()

+------+-------------+-------------+---+------+----------+------+------------+--------------------+
|emp_id|department_id|         Name|age|salary| hire_date|gender|Current_date|          Time_stamp|
+------+-------------+-------------+---+------+----------+------+------------+--------------------+
|   001|          101|     John Doe| 30| 50000|2015-01-01|     M|  2026-03-19|2026-03-19 14:36:...|
|   002|          101|   Jane Smith| 25| 45000|2016-02-15|     F|  2026-03-19|2026-03-19 14:36:...|
|   003|          102|    Bob Brown| 35| 55000|2014-05-01|     M|  2026-03-19|2026-03-19 14:36:...|
|   004|          102|    Alice Lee| 28| 48000|2017-09-30|     F|  2026-03-19|2026-03-19 14:36:...|
|   005|          103|    Jack Chan| 40| 60000|2013-04-01|     M|  2026-03-19|2026-03-19 14:36:...|
|   006|          103|    Jill Wong| 32| 52000|2018-07-01|     F|  2026-03-19|2026-03-19 14:36:...|
|   007|          101|James Johnson| 42| 70000|2012-03-15|     M|  2026-03-19|2026-03-19 14:36:...|


In [125]:
df_reordered.write.format("csv").save("data/output/4/emp.csv")

In [140]:
# convert date into string and print only date
from pyspark.sql.functions import date_format
df_final=df_reordered.withColumn("date_year",to_date(col("time_stamp"),"dd"))

In [141]:
df_final.show()

+------+-------------+-------------+---+------+----------+------+------------+--------------------+----------+
|emp_id|department_id|         Name|age|salary| hire_date|gender|Current_date|          Time_stamp| date_year|
+------+-------------+-------------+---+------+----------+------+------------+--------------------+----------+
|   001|          101|     John Doe| 30| 50000|2015-01-01|     M|  2026-03-19|2026-03-19 14:48:...|2026-03-19|
|   002|          101|   Jane Smith| 25| 45000|2016-02-15|     F|  2026-03-19|2026-03-19 14:48:...|2026-03-19|
|   003|          102|    Bob Brown| 35| 55000|2014-05-01|     M|  2026-03-19|2026-03-19 14:48:...|2026-03-19|
|   004|          102|    Alice Lee| 28| 48000|2017-09-30|     F|  2026-03-19|2026-03-19 14:48:...|2026-03-19|
|   005|          103|    Jack Chan| 40| 60000|2013-04-01|     M|  2026-03-19|2026-03-19 14:48:...|2026-03-19|
|   006|          103|    Jill Wong| 32| 52000|2018-07-01|     F|  2026-03-19|2026-03-19 14:48:...|2026-03-19|
|

In [142]:
emp_data_1 = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"]
]

emp_data_2 = [
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

In [153]:
emp_data_1=spark.createDataFrame(data=emp_data_1,schema=emp_schema)
emp_date_2=spark.createDataFrame(data=emp_data_2,schema=emp_schema)

TypeError: data is already a DataFrame

In [154]:
emp_date_2.show()

+-----------+-------------+-----------+---+------+------+----------+
|employee_id|department_id|       name|age|gender|salary| hire_date|
+-----------+-------------+-----------+---+------+------+----------+
|        011|          104| David Park| 38|  Male| 65000|2015-11-01|
|        012|          105| Susan Chen| 31|Female| 54000|2017-02-15|
|        013|          106|  Brian Kim| 45|  Male| 75000|2011-07-01|
|        014|          107|  Emily Lee| 26|Female| 46000|2019-01-01|
|        015|          106|Michael Lee| 37|  Male| 63000|2014-09-30|
|        016|          107|Kelly Zhang| 30|Female| 49000|2018-04-01|
|        017|          105|George Wang| 34|  Male| 57000|2016-03-15|
|        018|          104|  Nancy Liu| 29|      | 50000|2017-06-01|
|        019|          103|Steven Chen| 36|  Male| 62000|2015-08-01|
|        020|          102|  Grace Kim| 32|Female| 53000|2018-11-01|
+-----------+-------------+-----------+---+------+------+----------+



In [155]:
#union all

emp=emp_data_1.unionAll(emp_date_2)

In [156]:
emp.show()

+-----------+-------------+-------------+---+------+------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|
+-----------+-------------+-------------+---+------+------+----------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-08-01|
|        011|          104|   David Park| 38|  Male| 65000|2015-11-01|
|     

In [162]:
from pyspark.sql.functions import asc,col,desc
emp_sorted=emp.orderBy(col("salary").asc())

In [166]:
# Aggregation
# select dept_id, count(employee_id) as total_dept_count from emp_sorted group by dept_id 
from pyspark.sql.functions import count

emp_count=emp_sorted.groupBy("department_id").agg(count("employee_id").alias("total_dept_count"))
emp_count.show()

+-------------+----------------+
|department_id|total_dept_count|
+-------------+----------------+
|          101|               3|
|          102|               4|
|          103|               4|
|          104|               3|
|          105|               2|
|          107|               2|
|          106|               2|
+-------------+----------------+



In [168]:
# Aggregation
# select dept_id, sum(salary) as total_dept_salary from emp_sorted group by dept_id 
from pyspark.sql.functions import sum

emp_sum=emp_sorted.groupBy("department_id").agg(sum("salary").alias("total_dept_salary"))
emp_sum.show()

+-------------+-----------------+
|department_id|total_dept_salary|
+-------------+-----------------+
|          101|         165000.0|
|          107|          95000.0|
|          104|         162000.0|
|          102|         207000.0|
|          103|         232000.0|
|          106|         138000.0|
|          105|         111000.0|
+-------------+-----------------+



In [176]:
# Aggregation with having clause
# select dept_id, avg(salary) as avg_dept_salary from emp_sorted  group by dept_id having avg(salary) > 50000

from pyspark.sql.functions import avg

emp_avg=emp_sorted.groupBy("department_id").agg(avg("salary").alias("avg_dept_salary")).where("avg_dept_salary>50000")
emp_avg.show()

+-------------+---------------+
|department_id|avg_dept_salary|
+-------------+---------------+
|          101|        55000.0|
|          104|        54000.0|
|          102|        51750.0|
|          103|        58000.0|
|          106|        69000.0|
|          105|        55500.0|
+-------------+---------------+



In [179]:
# Bonus TIP - unionByName
# In case the column sequence is different
emp_fixed=emp_data_1.unionByName(emp_date_2)

In [180]:
emp_fixed.show()

+-----------+-------------+-------------+---+------+------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|
+-----------+-------------+-------------+---+------+------+----------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-08-01|
|        011|          104|   David Park| 38|  Male| 65000|2015-11-01|
|     

In [181]:
emp=spark.createDataFrame(data=emp_data,schema=emp_schema)

In [182]:
emp.show()

+-----------+-------------+-------------+---+------+------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|
+-----------+-------------+-------------+---+------+------+----------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-08-01|
|        011|          104|   David Park| 38|  Male| 65000|2015-11-01|
|     

In [191]:
from pyspark.sql.window import Window
from pyspark.sql.functions import max,col,desc

emp_unique=emp.distinct()
window_spec=Window.partitionBy(col("department_id")).orderBy(col("salary").desc())
max_func=max(col("salary")).over(window_spec)
emp_1=emp.withColumn("max_salary",max_func)
emp_1.show()

+-----------+-------------+-------------+---+------+------+----------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|max_salary|
+-----------+-------------+-------------+---+------+------+----------+----------+
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|     70000|
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|     70000|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|     70000|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|     55000|
|        020|          102|    Grace Kim| 32|Female| 53000|2018-11-01|     55000|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|     55000|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|     55000|
|        019|          103|  Steven Chen| 36|  Male| 62000|2015-08-01|     62000|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|     62000|
|        009|   

In [193]:
# 2nd highest salary each department
from pyspark.sql.functions import row_number

window_spec=Window.partitionBy(col("department_id")).orderBy(col("salary").desc())
rn=row_number().over(window_spec)
emp_2=emp.withColumn("rn",rn).where("rn=2")


In [195]:
emp_3=emp.withColumn("rn",expr("row_number() over (partition by department_id order by salary desc)")).where("rn=2")

In [196]:
emp_3.show()

+-----------+-------------+-----------+---+------+------+----------+---+
|employee_id|department_id|       name|age|gender|salary| hire_date| rn|
+-----------+-------------+-----------+---+------+------+----------+---+
|        001|          101|   John Doe| 30|  Male| 50000|2015-01-01|  2|
|        020|          102|  Grace Kim| 32|Female| 53000|2018-11-01|  2|
|        005|          103|  Jack Chan| 40|  Male| 60000|2013-04-01|  2|
|        018|          104|  Nancy Liu| 29|      | 50000|2017-06-01|  2|
|        012|          105| Susan Chen| 31|Female| 54000|2017-02-15|  2|
|        015|          106|Michael Lee| 37|  Male| 63000|2014-09-30|  2|
|        014|          107|  Emily Lee| 26|Female| 46000|2019-01-01|  2|
+-----------+-------------+-----------+---+------+------+----------+---+



In [197]:
dept_data = [
    ["101", "Sales", "NYC", "US", "1000000"],
    ["102", "Marketing", "LA", "US", "900000"],
    ["103", "Finance", "London", "UK", "1200000"],
    ["104", "Engineering", "Beijing", "China", "1500000"],
    ["105", "Human Resources", "Tokyo", "Japan", "800000"],
    ["106", "Research and Development", "Perth", "Australia", "1100000"],
    ["107", "Customer Service", "Sydney", "Australia", "950000"]
]

dept_schema = "department_id string, department_name string, city string, country string, budget string"

In [198]:
dept=spark.createDataFrame(data=dept_data,schema=dept_schema)

In [199]:
emp.show()
dept.show()

+-----------+-------------+-------------+---+------+------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|
+-----------+-------------+-------------+---+------+------+----------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-08-01|
|        011|          104|   David Park| 38|  Male| 65000|2015-11-01|
|     

In [200]:
emp.rdd.getNumPartitions()


4

In [201]:
dept.rdd.getNumPartitions()

4

In [206]:
emp_repartion=emp.repartition(3,"department_id")


In [207]:
emp_repartion.rdd.getNumPartitions()

3

In [208]:
from pyspark.sql.functions import spark_partition_id

emp_1=emp.repartition(3,"department_id").withColumn("partition_id",spark_partition_id())

In [209]:
emp_1.show()

+-----------+-------------+-------------+---+------+------+----------+------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|partition_id|
+-----------+-------------+-------------+---+------+------+----------+------------+
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-08-01|           0|
|        011|          104|   David Park| 38|  Male| 65000|2015-11-01|           0|
|        012|          105|   Susan Chen| 31|Female| 54000|2017-02-15|           0|
|        013|          106|    Brian Kim| 45|  Male| 75000|2011-07-01|           0|
|        015|          106|  Michael Lee| 37|  Male| 63000|2014-09-30|           0|
|        017|          105|  George Wang| 34|  Male| 57000|2016-03-15|           0|
|        018|          104|    Nancy Liu| 29|      | 50000|2017-06-01|           0|
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|           1|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|      

In [210]:
df_joined=emp.join(dept,how="inner",on=emp.department_id==dept.department_id)

In [211]:
df_joined.show()

+-----------+-------------+-------------+---+------+------+----------+-------------+--------------------+-------+---------+-------+
|employee_id|department_id|         name|age|gender|salary| hire_date|department_id|     department_name|   city|  country| budget|
+-----------+-------------+-------------+---+------+------+----------+-------------+--------------------+-------+---------+-------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|          101|               Sales|    NYC|       US|1000000|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|          101|               Sales|    NYC|       US|1000000|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|          101|               Sales|    NYC|       US|1000000|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|          102|           Marketing|     LA|       US| 900000|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|      

In [223]:
df_final=emp.join(dept,how="left_outer",on=(emp.department_id==dept.department_id) & ((emp.department_id=="101")|(emp.department_id=="102"))
                  &(emp.salary.isNull())
                 )

In [224]:
df_final.show()

+-----------+-------------+-------------+---+------+------+----------+-------------+---------------+----+-------+------+
|employee_id|department_id|         name|age|gender|salary| hire_date|department_id|department_name|city|country|budget|
+-----------+-------------+-------------+---+------+------+----------+-------------+---------------+----+-------+------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         null|           null|null|   null|  null|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         null|           null|null|   null|  null|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|         null|           null|null|   null|  null|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         null|           null|null|   null|  null|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         null|           null|null|   null|  null|
|        007|          101|James

In [228]:
df=spark.read.format("csv").option("header",True).option("inferschema",True).load("data/input/emp.csv")


In [229]:
df.printSchema()

root
 |-- employee_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- hire_date: timestamp (nullable = true)



In [230]:
_schema="employee_id int,department_id int,name string,age int,gender string,salary double,hire_date date"

In [231]:
df_schema=spark.read.format("csv").option("header",True).schema(_schema).load("data/input/emp.csv")

In [232]:
df_schema.show()

+-----------+-------------+-------------+---+------+-------+----------+
|employee_id|department_id|         name|age|gender| salary| hire_date|
+-----------+-------------+-------------+---+------+-------+----------+
|          1|          101|     John Doe| 30|  Male|50000.0|2015-01-01|
|          2|          101|   Jane Smith| 25|Female|45000.0|2016-02-15|
|          3|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|
|          4|          102|    Alice Lee| 28|Female|48000.0|2017-09-30|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|
|          6|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|
|         10|          104|     Lisa Lee| 27|Female|47000.0|2018-08-01|
|         11|          104|   David Park| 38|  Male|65000.0|2015

In [233]:
# permissive columnNameCorruptRecord
_schema="employee_id int,department_id int,name string,age int,gender string,salary double,hire_date date,bad_record string"
df_p=spark.read.format("csv").option("header",True).schema(_schema).option("columnNameofCorruptRecord",True).load("data/input/emp.csv")

In [234]:
df_p.show()

+-----------+-------------+-------------+---+------+-------+----------+----------+
|employee_id|department_id|         name|age|gender| salary| hire_date|bad_record|
+-----------+-------------+-------------+---+------+-------+----------+----------+
|          1|          101|     John Doe| 30|  Male|50000.0|2015-01-01|      null|
|          2|          101|   Jane Smith| 25|Female|45000.0|2016-02-15|      null|
|          3|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|      null|
|          4|          102|    Alice Lee| 28|Female|48000.0|2017-09-30|      null|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|      null|
|          6|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|      null|
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|      null|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|      null|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|      null|
|   

In [235]:
# handle bad record dropmalFormed

_schema=_schema="employee_id int,department_id int,name string,age int,gender string,salary double,hire_date date"
df_p=spark.read.format("csv").option("header",True).schema(_schema).option("mode","DropmalFormed").load("data/input/emp.csv")

In [236]:
df_p.show()

+-----------+-------------+-------------+---+------+-------+----------+
|employee_id|department_id|         name|age|gender| salary| hire_date|
+-----------+-------------+-------------+---+------+-------+----------+
|          1|          101|     John Doe| 30|  Male|50000.0|2015-01-01|
|          2|          101|   Jane Smith| 25|Female|45000.0|2016-02-15|
|          3|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|
|          4|          102|    Alice Lee| 28|Female|48000.0|2017-09-30|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|
|          6|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|
|         10|          104|     Lisa Lee| 27|Female|47000.0|2018-08-01|
|         11|          104|   David Park| 38|  Male|65000.0|2015

In [237]:
# handle bad record FAILFAST

_schema=_schema="employee_id int,department_id int,name string,age int,gender string,salary double,hire_date date"
df_p=spark.read.format("csv").option("header",True).schema(_schema).option("mode","FAILFAST").load("data/input/emp.csv")

In [238]:
df_p.show()

+-----------+-------------+-------------+---+------+-------+----------+
|employee_id|department_id|         name|age|gender| salary| hire_date|
+-----------+-------------+-------------+---+------+-------+----------+
|          1|          101|     John Doe| 30|  Male|50000.0|2015-01-01|
|          2|          101|   Jane Smith| 25|Female|45000.0|2016-02-15|
|          3|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|
|          4|          102|    Alice Lee| 28|Female|48000.0|2017-09-30|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|
|          6|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|
|          7|          101|James Johnson| 42|  Male|70000.0|2012-03-15|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|
|         10|          104|     Lisa Lee| 27|Female|47000.0|2018-08-01|
|         11|          104|   David Park| 38|  Male|65000.0|2015

In [242]:
#multiple_options
_options={
    "header":"true",
    "inferSchema":"true",
    "mode":"PERMISSIVE"
}
df=(spark.read.format("csv").options(**_options).load("data/input/emp.csv"))

In [244]:
#read Parquet
df_parquet=spark.read.format("parquet").load("data/input/sales_total_parquet/*.parquet")


In [ ]:
df_orc=spark.read.format("orc").load("data/input/sales_total_orc/*.orc")

In [ ]:
#check all files inside folder

df_1=spark.read.format("parquet").option("recursiveFileLookup",True).load("data/input/sales_recursive/")

In [ ]:

JSON = flexible, nested, unpredictable
{
  "emp_id": 1,
  "name": "Alice",
  "address": {              ← nested object!
    "city": "NYC",
    "zip": {                ← nested inside nested!
      "code": "10001",
      "area": "Manhattan"
    }
  },
  "skills": ["Python","SQL"] ← array!
  "projects": [             ← array of objects!
    {"id":1, "name":"ETL"},
    {"id":2, "name":"ML"}
  ]
}
Different records can have DIFFERENT fields!
Spark has to figure out structure! 😵

🎯 Types of JSON — Each Needs Different Approach!
Type 1 — JSON Lines (one record per line)
{"emp_id":1,"name":"Alice"}
{"emp_id":2,"name":"Bob"}
↑ Most common, easiest for Spark!

Type 2 — Pretty printed / Multiline
{
  "emp_id": 1,
  "name": "Alice"
}
↑ Needs special option!

Type 3 — JSON Array
[
  {"emp_id":1,"name":"Alice"},
  {"emp_id":2,"name":"Bob"}
]
↑ Spark can't read directly! Needs handling!

Type 4 — Nested JSON
{"emp_id":1,"address":{"city":"NYC"}}
↑ Needs explode/flattening!

Type 5 — Mixed/Inconsistent fields
{"emp_id":1,"name":"Alice","salary":80000}
{"emp_id":2,"name":"Bob"}              ← salary missing!
{"emp_id":3,"age":25}                  ← totally different field!
↑ Schema inference nightmare!

1️⃣ JSON Lines — Simplest!
python# data/emp.json
# {"emp_id":1,"name":"Alice","salary":80000}
# {"emp_id":2,"name":"Bob","salary":90000}

df = spark.read \
    .format("json") \
    .load("data/emp.json")

df.show()
# +------+-----+------+
# |emp_id| name|salary|
# +------+-----+------+
# |     1|Alice| 80000|
# |     2|  Bob| 90000|

df.printSchema()
# root
#  |-- emp_id: long
#  |-- name: string
#  |-- salary: long

2️⃣ Multiline JSON — Pretty Printed!
python# data/emp.json
# {
#   "emp_id": 1,
#   "name": "Alice"
# }

# ❌ Without multiLine — reads garbage!
df = spark.read.json("data/emp.json")
# each LINE read as separate record
# broken JSON on each line → nulls everywhere!

# ✅ With multiLine — reads correctly!
df = spark.read \
    .format("json") \
    .option("multiLine", "true") \   # ← KEY option!
    .load("data/emp.json")

df.show()
# +------+-----+
# |emp_id| name|
# +------+-----+
# |     1|Alice|

3️⃣ Nested JSON — The Real Challenge!
python# data/nested.json
# {"emp_id":1,"name":"Alice",
#  "address":{"city":"NYC","zip":"10001"},
#  "skills":["Python","SQL","Spark"]}

df = spark.read.json("data/nested.json")

df.printSchema()
# root
#  |-- emp_id: long
#  |-- name: string
#  |-- address: struct          ← nested struct!
#  |    |-- city: string
#  |    |-- zip: string
#  |-- skills: array            ← array!
#  |    |-- element: string

df.show(truncate=False)
# +------+-----+---------------+------------------+
# |emp_id|name |address        |skills            |
# +------+-----+---------------+------------------+
# |1     |Alice|{NYC, 10001}   |[Python, SQL, Spark]

# Access nested fields
df.select(
    "emp_id",
    "name",
    "address.city",        # dot notation for struct!
    "address.zip"
).show()
# +------+-----+----+-----+
# |emp_id| name|city|  zip|
# +------+-----+----+-----+
# |     1|Alice| NYC|10001|

4️⃣ Flatten Nested JSON — Production Pattern!
pythonfrom pyspark.sql.functions import col, explode

df = spark.read.json("data/nested.json")

# Flatten struct
df_flat = df.select(
    col("emp_id"),
    col("name"),
    col("address.city").alias("city"),    # flatten struct!
    col("address.zip").alias("zip")
)

# Explode array → one row per skill!
df_skills = df.select(
    col("emp_id"),
    col("name"),
    explode(col("skills")).alias("skill")  # array → rows!
)

df_skills.show()
# +------+-----+------+
# |emp_id| name| skill|
# +------+-----+------+
# |     1|Alice|Python|   ← one row per skill!
# |     1|Alice|   SQL|
# |     1|Alice| Spark|

5️⃣ Array of Objects — Complex Nested!
python# {"emp_id":1,
#  "projects":[
#    {"proj_id":1,"proj_name":"ETL"},
#    {"proj_id":2,"proj_name":"ML"}
#  ]}

df = spark.read.json("data/projects.json")

df.printSchema()
# |-- projects: array
# |    |-- element: struct
# |    |    |-- proj_id: long
# |    |    |-- proj_name: string

# Explode array of structs!
from pyspark.sql.functions import explode, col

df_exploded = df.select(
    col("emp_id"),
    explode(col("projects")).alias("project")  # array → rows
)

# Now access struct fields
df_final = df_exploded.select(
    col("emp_id"),
    col("project.proj_id"),
    col("project.proj_name")
)

df_final.show()
# +------+-------+---------+
# |emp_id|proj_id|proj_name|
# +------+-------+---------+
# |     1|      1|      ETL|  ← one row per project!
# |     1|      2|       ML|

6️⃣ Missing Fields — Handle Inconsistency!
python# {"emp_id":1,"name":"Alice","salary":80000}
# {"emp_id":2,"name":"Bob"}              ← no salary!
# {"emp_id":3,"salary":70000}            ← no name!

df = spark.read.json("data/inconsistent.json")

df.show()
# +------+-----+------+
# |emp_id| name|salary|
# +------+-----+------+
# |     1|Alice| 80000|
# |     2|  Bob|  null|  ← salary = null ✅
# |     3| null| 70000|  ← name = null ✅

# Spark fills missing fields with null automatically!

🎯 All Key JSON Options!
pythondf = spark.read \
    .format("json") \
    .option("multiLine",             "true") \  # pretty JSON
    .option("allowComments",         "true") \  # allow // comments
    .option("allowUnquotedFieldNames","true") \  # {name: "Alice"}
    .option("allowSingleQuotes",     "true") \  # {'name':'Alice'}
    .option("allowNumericLeadingZeros","true") \ # 007
    .option("mode", "PERMISSIVE") \             # nulls for bad rows
    .option("mode", "DROPMALFORMED") \          # skip bad rows
    .option("mode", "FAILFAST") \               # crash on bad row
    .option("columnNameOfCorruptRecord","_corrupt") \ # catch bad!
    .load("data/emp.json")